In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("Employee Attendance Tracker") \
    .getOrCreate()

In [3]:
# Upload CSV file
from google.colab import files

uploaded = files.upload()

Saving large_attendance.csv to large_attendance.csv


In [4]:
# Read uploaded CSV
df = spark.read.csv(
    "large_attendance.csv",
    header=True,
    inferSchema=True
)

print("Total Records:")
print(df.count())

print("Original Dataset")
df.show(10)

Total Records:
2000
Original Dataset
+-----------+----------+-------------------+-------------------+---------------+
|employee_id|department|           clock_in|          clock_out|tasks_completed|
+-----------+----------+-------------------+-------------------+---------------+
|          1|   Finance|2026-06-15 08:14:00|2026-06-15 14:14:00|              7|
|          2|        IT|2026-06-15 08:38:00|2026-06-15 13:38:00|             18|
|          3|   Finance|2026-06-15 10:10:00|2026-06-15 18:10:00|              3|
|          4|     Sales|2026-06-15 10:21:00|2026-06-15 18:21:00|              1|
|          5|     Sales|2026-06-15 09:37:00|2026-06-15 14:37:00|              0|
|          6|     Sales|2026-06-15 09:21:00|2026-06-15 17:21:00|             11|
|          7|        IT|2026-06-15 08:26:00|2026-06-15 14:26:00|              9|
|          8|     Sales|2026-06-15 10:46:00|2026-06-15 19:46:00|             18|
|          9|     Sales|2026-06-15 10:51:00|2026-06-15 14:51:00|        

In [7]:
# Convert datetime columns
df = df.withColumn(
    "clock_in",
    to_timestamp("clock_in")
)

df = df.withColumn(
    "clock_out",
    to_timestamp("clock_out")
)

In [9]:
# Calculate Work Hours
df = df.withColumn(
    "work_hours",
    (
        unix_timestamp("clock_out")
        -
        unix_timestamp("clock_in")
    ) / 3600
)


In [10]:
# Productivity Score
df = df.withColumn(
    "productivity_score",
    col("tasks_completed") /
    col("work_hours")
)

print("Dataset with Calculated Metrics")
df.show(10)

Dataset with Calculated Metrics
+-----------+----------+-------------------+-------------------+---------------+----------+------------------+
|employee_id|department|           clock_in|          clock_out|tasks_completed|work_hours|productivity_score|
+-----------+----------+-------------------+-------------------+---------------+----------+------------------+
|          1|   Finance|2026-06-15 08:14:00|2026-06-15 14:14:00|              7|       6.0|1.1666666666666667|
|          2|        IT|2026-06-15 08:38:00|2026-06-15 13:38:00|             18|       5.0|               3.6|
|          3|   Finance|2026-06-15 10:10:00|2026-06-15 18:10:00|              3|       8.0|             0.375|
|          4|     Sales|2026-06-15 10:21:00|2026-06-15 18:21:00|              1|       8.0|             0.125|
|          5|     Sales|2026-06-15 09:37:00|2026-06-15 14:37:00|              0|       5.0|               0.0|
|          6|     Sales|2026-06-15 09:21:00|2026-06-15 17:21:00|             11|

In [11]:

# Filter Late Logins (after 9:15 AM)
late_logins = df.filter(
    (hour("clock_in") > 9) |
    (
        (hour("clock_in") == 9) &
        (minute("clock_in") > 15)
    )
)

print("Late Login Count:")
print(late_logins.count())

late_logins.show(10)

Late Login Count:
1147
+-----------+----------+-------------------+-------------------+---------------+----------+------------------+
|employee_id|department|           clock_in|          clock_out|tasks_completed|work_hours|productivity_score|
+-----------+----------+-------------------+-------------------+---------------+----------+------------------+
|          3|   Finance|2026-06-15 10:10:00|2026-06-15 18:10:00|              3|       8.0|             0.375|
|          4|     Sales|2026-06-15 10:21:00|2026-06-15 18:21:00|              1|       8.0|             0.125|
|          5|     Sales|2026-06-15 09:37:00|2026-06-15 14:37:00|              0|       5.0|               0.0|
|          6|     Sales|2026-06-15 09:21:00|2026-06-15 17:21:00|             11|       8.0|             1.375|
|          8|     Sales|2026-06-15 10:46:00|2026-06-15 19:46:00|             18|       9.0|               2.0|
|          9|     Sales|2026-06-15 10:51:00|2026-06-15 14:51:00|              2|       4.

In [12]:
# Filter Absentees
absentees = df.filter(
    col("tasks_completed") == 0
)

print("Absentee Count:")
print(absentees.count())

absentees.show(10)

Absentee Count:
87
+-----------+----------+-------------------+-------------------+---------------+----------+------------------+
|employee_id|department|           clock_in|          clock_out|tasks_completed|work_hours|productivity_score|
+-----------+----------+-------------------+-------------------+---------------+----------+------------------+
|          5|     Sales|2026-06-15 09:37:00|2026-06-15 14:37:00|              0|       5.0|               0.0|
|         20|        IT|2026-06-15 08:14:00|2026-06-15 16:14:00|              0|       8.0|               0.0|
|         26|   Finance|2026-06-15 09:26:00|2026-06-15 15:26:00|              0|       6.0|               0.0|
|         52|     Sales|2026-06-15 10:51:00|2026-06-15 14:51:00|              0|       4.0|               0.0|
|         65|        IT|2026-06-15 10:02:00|2026-06-15 14:02:00|              0|       4.0|               0.0|
|         68|        HR|2026-06-15 10:26:00|2026-06-15 15:26:00|              0|       5.0|  

In [13]:
# Department Wise Summary
department_summary = df.groupBy(
    "department"
).agg(
    round(avg("work_hours"), 2)
    .alias("avg_work_hours"),

    round(avg("productivity_score"), 2)
    .alias("avg_productivity")
)

print("Department Summary")

department_summary.show()

Department Summary
+----------+--------------+----------------+
|department|avg_work_hours|avg_productivity|
+----------+--------------+----------------+
|     Sales|          6.48|             1.6|
|        HR|          6.48|            1.68|
|   Finance|          6.56|            1.61|
|        IT|          6.44|            1.59|
+----------+--------------+----------------+



In [14]:
# Top Departments by Productivity
department_summary.orderBy(
    col("avg_productivity").desc()
).show()

spark.stop()

+----------+--------------+----------------+
|department|avg_work_hours|avg_productivity|
+----------+--------------+----------------+
|        HR|          6.48|            1.68|
|   Finance|          6.56|            1.61|
|     Sales|          6.48|             1.6|
|        IT|          6.44|            1.59|
+----------+--------------+----------------+

